# Logistic Regression in PyTorch

Train a binary classifier on the **Breast Cancer Wisconsin (Diagnostic)** dataset:
569 samples and 30 numerical features computed from images of cell nuclei.
The sklearn labels are **0 — malignant, 1 — benign**. This is an educational experiment.
The dataset is bundled with sklearn; no download or network access is required.

Source: [UCI](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic).
Loader documentation: [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html).

**Goal:** understand logits, autograd, and the training loop; achieve test accuracy
of at least **0.95**; and compare your model with sklearn logistic regression.
Do not use hidden layers, change the data split, or tune settings on the test set.

Install dependencies with `pip install -r requirements.txt`. Use CPU throughout.
Complete the TODOs in this notebook. Before submitting,
restart the kernel and run every cell in order.


## 1. Theory

For a feature matrix $X\in\mathbb{R}^{N\times D}$, the model computes logits
$z=Xw+b$ and class-1 probabilities $p=\sigma(z)=1/(1+e^{-z})$.
The logit is linear in the features; classification uses $p\geq0.5$.

The mean binary cross-entropy is
$$L=-\frac1N\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)].$$
Its gradients are $\nabla_w L=X^T(p-y)/N$ and
$\partial L/\partial b=\operatorname{mean}(p-y)$.
PyTorch's `BCEWithLogitsLoss` combines sigmoid and BCE in a numerically stable way:
pass **logits**, not probabilities. Targets must have the same shape as the logits
and use a floating-point dtype.

`loss.backward()` accumulates gradients in `.grad`. Each step requires
`optimizer.zero_grad()`, a forward pass, loss computation, `backward()`, and `optimizer.step()`.
For evaluation, use `model.eval()` and `torch.no_grad()`; `eval()` alone does not
disable autograd. For `nn.Linear`, train/eval modes do not change predictions.

Standardization $x'=(x-\mu)/s$ helps optimization. Estimate $\mu,s$ on the training set only.
Use validation data to choose the learning rate and number of epochs; reserve test data for the final report.


In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

torch.set_num_threads(1)
torch.manual_seed(42)
np.random.seed(42)


## 2. Data (0.2 points)

Use the provided stratified 60/20/20 split. Inspect `dataset.DESCR`, split sizes,
and class proportions. Fit StandardScaler on the training set only and convert
all splits to float32 tensors. Targets must have shape `(N,)`.
Keep the names `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`.


In [ ]:
dataset = load_breast_cancer()
X_train_raw, X_holdout_raw, y_train_np, y_holdout = train_test_split(
    dataset.data, dataset.target, test_size=0.4, random_state=42,
    stratify=dataset.target)
X_val_raw, X_test_raw, y_val_np, y_test_np = train_test_split(
    X_holdout_raw, y_holdout, test_size=0.5, random_state=42,
    stratify=y_holdout)
print('Train / validation / test:', len(y_train_np), len(y_val_np), len(y_test_np))
print('Classes:', dict(enumerate(dataset.target_names)))


In [ ]:
# TODO: standardize features and convert to tensors.


In [ ]:
assert X_train.shape == (341, 30)
assert y_train.shape == (341,)
assert X_train.dtype == y_train.dtype == torch.float32
assert torch.allclose(X_train.mean(0), torch.zeros(30), atol=1e-5)


## 3. Model and training (0.4 points)

Complete the model below. Use a single `nn.Linear(30, 1)` layer with bias.
The `forward` method must return logits of shape `(N,)`. Remove only the last
dimension with `squeeze(-1)` so that a batch containing one sample also works.

Train with full-batch SGD, no regularization, and `BCEWithLogitsLoss()`.
`train_model` must return four lists of metrics computed after each update.
Implement probability and label prediction in `predict_proba` and `predict`.

In [ ]:
class LogisticRegression(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        # TODO: one linear layer with bias and a single output.
        raise NotImplementedError()

    def forward(self, X):
        """X: float32 (N, D). Return logits of shape (N,), without sigmoid."""
        # TODO
        raise NotImplementedError()


def train_model(model, X_train, y_train, X_val, y_val, *, lr=0.1, epochs=500):
    """Train the supplied model with full-batch SGD and BCEWithLogitsLoss.

    All tensors are on CPU. Features are float32; targets are float32 of shape (N,).
    Return a dictionary with train_loss, val_loss, train_accuracy, and val_accuracy
    lists, each of length epochs. Compute all metrics AFTER each parameter update,
    without building a computation graph. Accuracy uses probability threshold 0.5.
    Loss is mean BCE without regularization. Do not recreate the model inside.
    """
    # TODO
    raise NotImplementedError()


def predict_proba(model, X):
    """Return class-1 probabilities: a tensor of shape (N,) without a graph."""
    # TODO
    raise NotImplementedError()


def predict(model, X, threshold=0.5):
    """Return int64 labels (N,): probability >= threshold means class 1."""
    # TODO
    raise NotImplementedError()


In [ ]:
torch.manual_seed(42)
model = LogisticRegression(X_train.shape[1])
# Start with lr=0.1 and epochs=500. Choose settings using validation only.
history = train_model(model, X_train, y_train, X_val, y_val, lr=0.1, epochs=500)
assert model(X_train[:1]).shape == (1,)
assert set(history) == {'train_loss', 'val_loss', 'train_accuracy', 'val_accuracy'}


## 4. Learning curves and final evaluation (0.2 points)

Plot two panels: training/validation loss and training/validation accuracy versus epoch.
Explain your observations. Once you have selected the settings, freeze them and
evaluate on the test set once. Compute accuracy and a confusion matrix with class order `[0, 1]`.
Compare against a constant classifier that predicts the most frequent **training** class.
Required performance: **test accuracy ≥ 0.95**. Also report the number of errors
and test-set size: accuracy changes in discrete steps on a small dataset.


In [ ]:
import matplotlib.pyplot as plt
# TODO: plot the training history.


In [ ]:
# Run after freezing the training settings.
test_predictions = predict(model, X_test).cpu().numpy()
test_accuracy = accuracy_score(y_test_np, test_predictions)
print('Test accuracy:', test_accuracy)
print(confusion_matrix(y_test_np, test_predictions, labels=[0, 1]))
# TODO: compute baseline accuracy and the number of errors.
assert test_accuracy >= 0.95


## 5. Comparison with sklearn (0.2 points)

Train `sklearn.linear_model.LogisticRegression(penalty=None, solver="lbfgs",
max_iter=5000, tol=1e-8)` on the same standardized training features.
Disable regularization to match the unpenalized BCE objective in PyTorch.
For sklearn 1.9+, use `C=float("inf")` instead of `penalty=None`.
Do not refit the scaler or add validation samples to the training set.

Create a table of training/validation/test accuracy for both models. On the test set,
count prediction disagreements and compute the mean absolute difference between
class-1 probabilities (`predict_proba(...)[:, 1]` in sklearn).
Explain why results may differ: finite-step SGD and LBFGS have different stopping
conditions. Matching weights is not required.
Test results are for reporting only: do not tune either model after comparing them.


In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
# TODO: train sklearn, build the table, and compare predictions.


## 6. Optional. You may add more layers to your pytorch classifier (and make it a simple NN) to improve the accuracy. Significant improvement with explanation will bring additional points (up to 0.5).

**Conclusions:** TODO

## Submission

Before submitting, restart the kernel and run every cell in order.